In [1]:
import numpy as np
import pandas as pd

distance_matrix = np.load('../data/distance_matrix.npy').astype(int)
sample = pd.read_csv('../data/sample_stops.csv')
routes_df = pd.read_csv('../data/solution_routes.csv')

N_VEHICLES = 4
CAPACITY = 130
DEPOT = 0

demand = sample['demand'].to_numpy()
print(f"{len(sample)-1} stops, {demand.sum()} packages, "
      f"{N_VEHICLES} vehicles x {CAPACITY} = {N_VEHICLES*CAPACITY} capacity")

150 stops, 466 packages, 4 vehicles x 130 = 520 capacity


In [2]:
def route_distance(seq):
    """Total meters for one route. seq must start and end at the depot."""
    return sum(distance_matrix[seq[i]][seq[i+1]] for i in range(len(seq) - 1))

def fleet_distance(routes):
    """Total meters across all routes, in km."""
    return sum(route_distance(r) for r in routes) / 1000

In [3]:
def baseline_dataset_order():
    """Stops in county-file order, packed into trucks until capacity is hit."""
    routes, current, load = [], [DEPOT], 0

    for node in range(1, len(sample)):
        d = int(demand[node])
        if load + d > CAPACITY:
            routes.append(current + [DEPOT])
            current, load = [DEPOT], 0
        current.append(node)
        load += d

    routes.append(current + [DEPOT])
    return routes

routes_naive = baseline_dataset_order()
print(f"drivers used: {len(routes_naive)}")
print(f"total: {fleet_distance(routes_naive):.2f} km")

drivers used: 4
total: 1836.97 km


In [4]:
def baseline_nearest_neighbor():
    """From each position, drive to the closest stop that still fits."""
    unvisited = set(range(1, len(sample)))
    routes = []

    for _ in range(N_VEHICLES):
        current, load, position = [DEPOT], 0, DEPOT

        while True:
            feasible = [n for n in unvisited if load + demand[n] <= CAPACITY]
            if not feasible:
                break
            nxt = min(feasible, key=lambda n: distance_matrix[position][n])
            current.append(nxt)
            load += int(demand[nxt])
            unvisited.remove(nxt)
            position = nxt

        routes.append(current + [DEPOT])

    if unvisited:
        print(f"WARNING: {len(unvisited)} stops unserved by nearest neighbor")
    return routes

routes_nn = baseline_nearest_neighbor()
print(f"drivers used: {len(routes_nn)}")
print(f"total: {fleet_distance(routes_nn):.2f} km")

drivers used: 4
total: 395.79 km


In [5]:
routes_opt = [
    grp.sort_values('stop_sequence')['node'].tolist()
    for _, grp in routes_df.groupby('vehicle')
]

for r in routes_opt:
    assert r[0] == DEPOT and r[-1] == DEPOT, "route doesn't start/end at depot"

print(f"drivers used: {len(routes_opt)}")
print(f"total: {fleet_distance(routes_opt):.2f} km")

drivers used: 3
total: 145.53 km


In [8]:
def validate(name, routes):
    problems = []
    visited = []

    for r in routes:
        if r[0] != DEPOT or r[-1] != DEPOT:
            problems.append("route doesn't start and end at the depot")
        load = sum(int(demand[n]) for n in r[1:-1])
        if load > CAPACITY:
            problems.append(f"load {load} exceeds capacity {CAPACITY}")
        visited.extend(r[1:-1])

    dupes = len(visited) - len(set(visited))
    missing = set(range(1, len(sample))) - set(visited)

    if dupes:
        problems.append(f"{dupes} stop(s) visited more than once")
    if missing:
        problems.append(f"{len(missing)} stop(s) never visited")
    if len(routes) > N_VEHICLES:
        problems.append(f"uses {len(routes)} drivers, fleet is {N_VEHICLES}")

    flag = "OK  " if not problems else "FAIL"
    print(f"{flag} {name:24} {len(set(visited)):3} stops, {len(routes)} routes")
    for p in problems:
        print(f"       - {p}")
    return not problems


ok = all([
    validate('Dataset order', routes_naive),
    validate('Nearest neighbor', routes_nn),
    validate('OR-Tools', routes_opt),
])
assert ok, "a route set failed validation — table would be misleading"

OK   Dataset order            150 stops, 4 routes
OK   Nearest neighbor         150 stops, 4 routes
FAIL OR-Tools                  20 stops, 3 routes
       - 130 stop(s) never visited


AssertionError: a route set failed validation — table would be misleading

In [6]:
KM_PER_MILE = 1.60934
COST_PER_MILE = 0.70      # assumption: fully-burdened light-duty van operating cost
AVG_SPEED_KMH = 30        # assumption: urban delivery average, incl. stops/lights
SERVICE_MIN = 5           # assumption: minutes per delivery

def summarize(name, routes):
    km = fleet_distance(routes)
    stops = sum(len(r) - 2 for r in routes)
    drive_hours = km / AVG_SPEED_KMH
    service_hours = stops * SERVICE_MIN / 60
    return {
        'Method': name,
        'Drivers': len(routes),
        'Distance (km)': round(km, 1),
        'Distance (mi)': round(km / KM_PER_MILE, 1),
        'Driver-hours': round(drive_hours + service_hours, 2),
        'Daily cost ($)': round(km / KM_PER_MILE * COST_PER_MILE, 2),
    }

comparison = pd.DataFrame([
    summarize('Dataset order (no tool)', routes_naive),
    summarize('Nearest neighbor', routes_nn),
    summarize('OR-Tools (GLS)', routes_opt),
])

base_km = comparison.loc[0, 'Distance (km)']
comparison['vs. no tool'] = comparison['Distance (km)'].apply(
    lambda km: f"{(base_km - km) / base_km * 100:+.1f}%"
)

comparison

,Method,Drivers,Distance (km),Distance (mi),Driver-hours,Daily cost ($),vs. no tool
0,Dataset order (no tool),4,1837.0,1141.4,73.73,799.01,+0.0%
1,Nearest neighbor,4,395.8,245.9,25.69,172.15,+78.5%
2,OR-Tools (GLS),3,145.5,90.4,6.52,63.30,+92.1%


In [7]:
naive_km = comparison.loc[0, 'Distance (km)']
nn_km    = comparison.loc[1, 'Distance (km)']
opt_km   = comparison.loc[2, 'Distance (km)']

pct_vs_naive = (naive_km - opt_km) / naive_km * 100
pct_vs_nn    = (nn_km - opt_km) / nn_km * 100
annual = (naive_km - opt_km) / KM_PER_MILE * COST_PER_MILE * 260

print(f"vs. dataset order:   {pct_vs_naive:.1f}% fewer km")
print(f"vs. nearest neighbor: {pct_vs_nn:.1f}% fewer km")
print(f"annualized savings (260 operating days): ${annual:,.0f}\n")

if pct_vs_naive < 5:
    print("*** WARNING: optimizer barely beats the naive baseline.")
    print("*** The arc cost evaluator is probably still not registered.")
    print("*** Re-check that cell in 05 is a CODE cell and re-run all.")

comparison.to_csv('../output/comparison.csv', index=False)

vs. dataset order:   92.1% fewer km
vs. nearest neighbor: 63.2% fewer km
annualized savings (260 operating days): $191,291

